# Download Fruit Dataset from Kaggle

This notebook downloads `sriramr/fruits-fresh-and-rotten-for-classification` and prepares the fresh fruit folders expected by the project:

- `dataset/train/freshapples/`
- `dataset/train/freshoranges/`
- `dataset/train/freshbananas/`
- `dataset/test/freshapples/`
- `dataset/test/freshoranges/`
- `dataset/test/freshbananas/`

Before running, download your Kaggle API token from `https://www.kaggle.com/settings` and save it as:

```text
C:\Users\Rafael Po\.kaggle\kaggle.json
```

## 1. Install Kaggle API

Run this cell once. If `kaggle` is already installed, it will simply confirm the package is available.

In [1]:
%pip install kaggle

Note: you may need to restart the kernel to use updated packages.


## 2. Configure Paths

These paths are set for this project and your Windows username.

In [2]:
from pathlib import Path

DATASET_SLUG = "sriramr/fruits-fresh-and-rotten-for-classification"

USER_HOME = Path(r"/Users/theofilius")
KAGGLE_JSON = USER_HOME / ".kaggle" / "kaggle.json"

PROJECT_ROOT = Path(r"/Users/theofilius/Downloads/fruit-grading-model-main")
RAW_DIR = PROJECT_ROOT / "raw"
DATASET_ROOT = PROJECT_ROOT / "dataset"

print(f"Kaggle credentials: {KAGGLE_JSON}")
print(f"Raw download folder: {RAW_DIR}")
print(f"Prepared dataset folder: {DATASET_ROOT}")

Kaggle credentials: /Users/theofilius/.kaggle/kaggle.json
Raw download folder: /Users/theofilius/Downloads/fruit-grading-model-main/raw
Prepared dataset folder: /Users/theofilius/Downloads/fruit-grading-model-main/dataset


## 3. Validate Kaggle Credentials

In [3]:
if not KAGGLE_JSON.exists():
    raise FileNotFoundError(
        "Kaggle API token not found. Download kaggle.json from "
        "https://www.kaggle.com/settings and place it at: "
        f"{KAGGLE_JSON}"
    )

RAW_DIR.mkdir(parents=True, exist_ok=True)
DATASET_ROOT.mkdir(parents=True, exist_ok=True)

print("Kaggle credentials found.")

Kaggle credentials found.


## 4. Download and Extract Dataset

This downloads the Kaggle zip into `raw/` and extracts it there.

In [4]:
import zipfile

from kaggle.api.kaggle_api_extended import KaggleApi

api = KaggleApi()
api.authenticate()

api.dataset_download_files(
    DATASET_SLUG,
    path=str(RAW_DIR),
    unzip=False,
)

zip_path = RAW_DIR / "fruits-fresh-and-rotten-for-classification.zip"
extract_dir = RAW_DIR / "fruits-fresh-and-rotten-for-classification"

if not zip_path.exists():
    raise FileNotFoundError(f"Expected zip file was not found: {zip_path}")

if not extract_dir.exists():
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extract_dir)
    print(f"Extracted dataset to: {extract_dir}")
else:
    print(f"Dataset already extracted at: {extract_dir}")

Dataset URL: https://www.kaggle.com/datasets/sriramr/fruits-fresh-and-rotten-for-classification
Extracted dataset to: /Users/theofilius/Downloads/fruit-grading-model-main/raw/fruits-fresh-and-rotten-for-classification


## 5. Copy Fresh Fruit Folders into `dataset/`

Only fresh apples, oranges, and bananas are copied. Rotten fruit folders are ignored.

In [5]:
import shutil

source_root = extract_dir / "dataset"
if not source_root.exists():
    source_root = extract_dir

fresh_folders = ["freshapples", "freshoranges", "freshbanana"]
splits = ["train", "test"]

for split in splits:
    for fruit_folder in fresh_folders:
        src = source_root / split / fruit_folder
        dst = DATASET_ROOT / split / fruit_folder

        if not src.exists():
            raise FileNotFoundError(f"Expected source folder was not found: {src}")

        dst.parent.mkdir(parents=True, exist_ok=True)

        if dst.exists():
            shutil.rmtree(dst)

        shutil.copytree(src, dst)

print("Fresh fruit folders copied into dataset/.")

Fresh fruit folders copied into dataset/.


## 6. Verify Dataset Structure

In [6]:
image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

print("Dataset ready:")
for split in splits:
    for fruit_folder in fresh_folders:
        folder = DATASET_ROOT / split / fruit_folder
        image_count = sum(1 for file in folder.iterdir() if file.suffix.lower() in image_extensions)
        print(f"{split}/{fruit_folder}: {image_count} images")

Dataset ready:
train/freshapples: 1693 images
train/freshoranges: 1466 images
train/freshbanana: 1581 images
test/freshapples: 395 images
test/freshoranges: 388 images
test/freshbanana: 381 images
